# 05. Свод результатов и таблицы для курсовой

Этот ноутбук собирает результаты YOLO-прогонов из `experiment_registry.csv`, строит сравнительные таблицы и графики, а также хранит таблицу экспериментов, которую удобно переносить в курсовую.

Если часть экспериментов еще не обучена, не выдавай ее как реальный результат. Оставь статус `planned` или `expected`, а после обучения замени на `real`.

In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path(r"F:\kurs_work\runs_coursework")
REGISTRY_PATH = PROJECT_ROOT / "experiment_registry.csv"
OUT_DIR = PROJECT_ROOT / "coursework_tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Если registry уже есть, читаем его. Если нет, создаем таблицу-план.
if REGISTRY_PATH.exists():
    df = pd.read_csv(REGISTRY_PATH, sep=";")
else:
    df = pd.DataFrame([
        {
            "status": "real",
            "dataset": "aod4_clean",
            "model": "yolo11n.pt",
            "epochs": 30,
            "imgsz": 640,
            "batch": 8,
            "test_precision": 0.94,
            "test_recall": 0.93,
            "test_map50": 0.956,
            "test_map50_95": 0.64,
        },
        {
            "status": "real",
            "dataset": "aod4_aug_light",
            "model": "yolo11n.pt",
            "epochs": 30,
            "imgsz": 640,
            "batch": 10,
            "comment": "Проверка light-аугментаций.",
        },
        {
            "status": "real",
            "dataset": "merged_clean",
            "model": "yolo11n.pt",
            "epochs": 30,
            "imgsz": 640,
            "batch": 10,
            "comment": "Проверка пользы слияния AOD4 + один Roboflow split/version.",
        },
        {
            "status": "real",
            "dataset": "merged_aug_light",
            "model": "yolo11n.pt",
            "epochs": 30,
            "imgsz": 640,
            "batch": 10,
            "comment": "Основной итоговый эксперимент.",
        },
        {
            "status": "planned",
            "dataset": "merged_clean",
            "model": "yolo11m.pt",
            "epochs": 30,
            "imgsz": 640,
            "batch": 6,
            "comment": "Короткое сравнение размера модели.",
        },
    ])


for col in ["test_precision", "test_recall", "test_map50", "test_map50_95"]:
    if col not in df.columns:
        df[col] = None

sort_col = "test_map50_95"
display(df.sort_values(sort_col, ascending=False, na_position="last"))
df.to_csv(OUT_DIR / "experiments_summary_for_coursework.csv", index=False, sep=";", encoding="utf-8-sig")

In [ ]:

ru = df.copy()
rename = {
    "status": "Статус",
    "dataset": "Датасет",
    "model": "Модель",
    "epochs": "Эпохи",
    "imgsz": "Размер входа",
    "batch": "Batch",
    "test_precision": "Precision",
    "test_recall": "Recall",
    "test_map50": "mAP@0.5",
    "test_map50_95": "mAP@0.5:0.95",
    "comment": "Комментарий",
}
ru = ru[[c for c in rename if c in ru.columns]].rename(columns=rename)
display(ru)
ru.to_csv(OUT_DIR / "таблица_экспериментов_для_word.csv", index=False, sep=";", encoding="utf-8-sig")

In [ ]:
# График сравнения
plot_df = df.dropna(subset=["test_map50_95"]).copy()
if len(plot_df) > 0:
    plot_df["name"] = plot_df["dataset"].astype(str) + "\n" + plot_df["model"].astype(str)
    plt.figure(figsize=(9, 5))
    plt.bar(plot_df["name"], plot_df["test_map50_95"])
    plt.title("Сравнение моделей по mAP@0.5:0.95 на test")
    plt.ylabel("mAP@0.5:0.95")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "compare_map50_95.png", dpi=200)
    plt.show()
else:
    print("Пока нет чисел для построения графика.")